# Gemma4EC — Predict EC Numbers

This notebook walks you through predicting EC numbers from reaction SMILES using Gemma4EC.

## Steps
1. Reaction SMILES are formatted into prompts
2. A pretrained LLM (TxGemma) is loaded from Hugging Face
3. An adapter file is layered on top - trained specifically for EC prediction
4. The model generates EC number predictions for each reaction

## Before running
- [ ] You have the `gemma4ec` conda environment activated
- [ ] You have requested access to TxGemma on Hugging Face ([9B](https://huggingface.co/google/txgemma-9b-predict) or [2B](https://huggingface.co/google/txgemma-2b-predict))
- [ ] You have downloaded the adapter: `git clone https://huggingface.co/PlanesLab/Gemma4EC-9B-Predict models/Gemma4EC-9B-Predict`
- [ ] You have a GPU with ≥18 GB VRAM (9B model) or ≥6 GB (2B model)

## 1. Authenticate with Hugging Face

TxGemma is a gated model — you need a Hugging Face account and to have requested access.
Run this once and your token will be saved locally (you won't need to do this again).

In [ ]:
from huggingface_hub import login

# This opens an interactive prompt asking for your HF token.
# Get your token at: https://huggingface.co/settings/tokens
login()

## 2. Configure paths

These defaults match the repo structure — **you only need to change `ADAPTER_FOLDER`** to point to where you downloaded the adapter.
If you put it in `models/Gemma4EC-9B-Predict` as suggested in the README, you don't need to change anything.

In [ ]:
# ── Only change these if your setup differs from the defaults ─────────────────

ADAPTER_FOLDER   = "./models/Gemma4EC-9B-Predict"     # downloaded LoRA adapter
MODEL_ID         = "google/txgemma-9b-predict"         # or txgemma-2b-predict

INPUT_CSV        = "./reactions.csv"                   # your input (see cell below)
OUTPUT_DIR       = "./outputs/results"                 # predictions saved here
PROCESSED_DIR    = "./outputs/inter_data"              # intermediate files
PROMPTS_JSON     = "./prompt_examples/prompts.json"
CACHE_DIR        = "./hfcache"                         # HF model cache
SAVE_DIR         = "./local_txgemma_model"             # where the base model is stored locally

# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))  # make sure repo root is on path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

print("✓ Paths configured")
print(f"  Adapter : {ADAPTER_FOLDER}")
print(f"  Model   : {MODEL_ID}")
print(f"  Output  : {OUTPUT_DIR}")

## 3. Prepare input reactions

Your input must be a CSV with a `rxn_smiles` column.
The cell below creates a small example — replace it with `pd.read_csv('your_file.csv')` to use your own data.

In [ ]:
import pandas as pd

# ── Option A: use example reactions ───────────────────────────────────────────
df = pd.DataFrame({
    "rxn_smiles": [
        "OC(=O)CCC(=O)C(=O)O.OC(=O)CCC(N)C(=O)O>>OC(=O)CCC(=O)C(=O)O",
        "CC(=O)SCCNC(=O)CCNC(=O)C(O)C(C)(C)COP(O)(=O)OP(O)(=O)OCC1OC(n2cnc3c(N)ncnc23)C(O)C1O>>CC(O)C(C)(C)COP(O)(=O)OP(O)(=O)OCC1OC(n2cnc3c(N)ncnc23)C(O)C1O",
    ]
})

# ── Option B: load your own CSV ───────────────────────────────────────────────
# df = pd.read_csv("path/to/your_reactions.csv")
# assert "rxn_smiles" in df.columns, "CSV must have a 'rxn_smiles' column"

df.to_csv(INPUT_CSV, index=False)
print(f"✓ {len(df)} reaction(s) ready")
df

## 4. Format reactions into prompts

Gemma4EC expects reactions wrapped in a specific prompt template.
This step converts your CSV into a JSONL file with the correct format.

In [ ]:
from src.datamod.templates import build_input_template_from_file
from src.datamod.jsonl_builder import create_jsonl_from_df
from src.utils.paths import get_jsonl_path

TEMPLATE_KEY   = "EC_NUMBER_PROMPT_2"   # recommended — do not change unless retraining
TEMPLATE_ALIAS = "EC_NUMBER"

input_template = build_input_template_from_file(
    template_path=PROMPTS_JSON,
    key=TEMPLATE_KEY,
    required_vars={"reaction_smiles"},
    dict_key=TEMPLATE_ALIAS,
)

jsonl_path = get_jsonl_path(PROCESSED_DIR, "predict", TEMPLATE_ALIAS, TEMPLATE_KEY)
create_jsonl_from_df(INPUT_CSV, jsonl_path, input_template)

print(f"✓ Prompts written to: {jsonl_path}")

## 5. Load the model and adapter

This step downloads TxGemma from Hugging Face (first run only — ~18 GB for 9B, ~4 GB for 2B)
and loads the LoRA adapter on top.

> **This will take a few minutes on first run** while the model downloads. Subsequent runs load from cache instantly.

In [ ]:
import os
os.environ.setdefault("WORLD_SIZE", "1")
os.environ.setdefault("RANK", "0")
os.environ.setdefault("LOCAL_RANK", "0")
os.environ.setdefault("MASTER_ADDR", "localhost")
os.environ.setdefault("MASTER_PORT", "12355")

from accelerate import Accelerator
from src.models.modeling import build_model_and_tokenizer
from src.training.manual.checkpoint_utils import reload_best_checkpoint

accelerator = Accelerator()

print(f"Loading base model: {MODEL_ID} ...")
model, tokenizer = build_model_and_tokenizer(MODEL_ID, SAVE_DIR, CACHE_DIR)

print(f"Loading adapter: {ADAPTER_FOLDER} ...")
model = reload_best_checkpoint(model, accelerator, ADAPTER_FOLDER)

print("✓ Model ready")

## 6. Predict

Run inference on your reactions. Results are saved as a TSV file in `OUTPUT_DIR`.

In [ ]:
import types
from datasets import load_dataset
from src.evaluation.eval import evaluate_model

data = load_dataset("json", data_files={"pred": jsonl_path})

args = types.SimpleNamespace(
    output_folder=OUTPUT_DIR,
    eval_batch_size=8,          # reduce further if you run out of VRAM
    template_key=TEMPLATE_KEY,
    template_alias=TEMPLATE_ALIAS,
    model_id=MODEL_ID,
    few_shot_trained=False,
)

evaluate_model(
    accelerator, model, tokenizer,
    data["pred"], args,
    split="pred",
    save_predictions=True,
)

print(f"\n✓ Done! Predictions saved to: {OUTPUT_DIR}")

## 7. View results

The output file is named: `predictions_pred.EC_NUMBER.EC_NUMBER_PROMPT_2.<timestamp>.tsv`

It contains these columns:
- `pred_raw` — the raw EC number predicted by the model
- `pred_fragment` — leading digits/dots extracted (useful if the model outputs extra text)
- `pred_well_formed` — whether the prediction is a valid EC format (True/False)
- `true_label` — ground truth, if provided in the input CSV
- `smiles` — the input reaction SMILES

In [ ]:
import glob

tsv_files = glob.glob(f"{OUTPUT_DIR}/*.tsv")

if tsv_files:
    results = pd.read_csv(tsv_files[0], sep="\t")
    print(f"✓ {len(results)} predictions loaded from {tsv_files[0]}")
    display(results)
else:
    print(f"No TSV found in {OUTPUT_DIR} — check that the previous cell ran without errors.")